<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 12: Latent Diffusion and Generative Image Models

#### Tim Moroney, 2026

Today we will complete our journey through the latent diffusion topic, by building our own generative image model.  Not for cats though, that would be a bit much to train on a CPU (same concepts though), plus we don't have a giant training set of cat images handy.  Instead, we'll return to our favourite MNIST dataset, and build a generative _digit_ image model.

# Package management

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# CVAE

We need our CVAE model to encode and decode latents.  We already have a suitable architecture from lesson 9.  Here are the encoder and decoder from that lesson, with just one architectural change: we have added an additional dense layer in the `mu` and `logvar` chains.  This is just to help the CVAE generate the nicest possible latent space for us to use in our reverse diffusion.

In [ ]:
function cvae_encoder(latent_dim)

    return @compact(
        embed = Chain(
                Conv((3, 3), 1 => 16; stride=2, pad=1), BatchNorm(16, swish),
                Conv((3, 3), 16 => 32; stride=2, pad=1), BatchNorm(32, swish),
                Conv((3, 3), 32 => 64; stride=2, pad=1), BatchNorm(64, swish),
                FlattenLayer()
        ),
        mu = Chain(Dense(1024 => 1024, swish), Dense(1024 => latent_dim)),
        logvar = Chain(Dense(1024 => 1024, swish), Dense(1024 => latent_dim))
    ) do x
        y = embed(x)
        μ = mu(y)
        logσ² = clamp.(logvar(y), -20.0f0, 10.0f0)
        σ = exp.(logσ² .* 0.5f0)
        ϵ = randn_like(σ)
        z = μ .+ σ .* ϵ

        @return z, μ, logσ²
    end
end

In [ ]:
function cvae_decoder(latent_dim)
    return @compact(

        seed = Chain(Dense(latent_dim => 1024, swish), Dense(1024 => 1024)),

        upchain = Chain(
                Upsample(2), Conv((3, 3), 64 => 32; stride = 1, pad = 1), BatchNorm(32, swish),
                Upsample(2), Conv((3, 3), 32 => 16; stride = 1, pad = 1), BatchNorm(16, swish),
                Upsample(2), Conv((3, 3), 16 => 1, sigmoid; stride = 1, pad = 1)
        )
    ) do z
        y = seed(z)
        img = reshape(y, 4, 4, 64, :)
        @return upchain(img)
    end
end

#
Previously we trained this model with a latent dimension of 2, so that we could visualise where the digits map to in latent space.  Today we want a richer latent representation since it's just one part of a larger model.  So we'll opt, rather arbitrarily, for a latent dimension of 8.  Rather than train it again, we'll load some pre-trained parameters.

In [ ]:
# CVAE architecture
latent_dim = 8
encoder = cvae_encoder(latent_dim)
decoder = cvae_decoder(latent_dim)

# Pre-trained parameters
url = "https://github.com/moroneyt/MXB301/raw/main/resources/CVAE_8d_params.jld2"
paramfile = jldopen(download(url))
p_cvae  = paramfile["p"]
st_cvae = paramfile["st"];

#
We'll also want some helper functions for doing the encoding and decoding, remembering to turn on test mode, so that the batch norm layers use the running means and variances during inference.

In [ ]:
# Helper function for encoding
function encode(image)
    (z, μ, logσ²), _ = encoder(image[:,:,:,:], p_cvae.encoder, Lux.testmode(st_cvae.encoder))
    return μ
end

# Helper function for decoding
function decode(z)
    image, _ = decoder(z, p_cvae.decoder, Lux.testmode(st_cvae.decoder))
    return image
end

#
We load the $32 \times 32$ MNIST dataset and convert pixel intensities to $[0,1]$ as usual.

In [ ]:
# Load the MNIST datset
url = "https://github.com/moroneyt/MXB301/raw/main/resources/MNIST32.jld2"
MNISTfile = jldopen(download(url))
images = MNISTfile["images"] / 255f0
labels = MNISTfile["labels"];

# Ornstein-Uhlenbeck process in $\mathbb{R}^d$

In the last lesson, we studied the SDE for a (scalar) random process $Z_t$

$$
\textrm{d}Z_{t} = f(Z_t,t)\, \textrm{d}t + \sigma\, \textrm{d}W_t\qquad
$$

and its associated Fokker-Planck equation for the probability density $u(z,t)$

$$
\frac{\partial u}{\partial t} = -\frac{\partial}{\partial z}(f u) + D \frac{\partial^2 u} {\partial z^2}
$$

where $D = \sigma^2 /2$.

We will now treat $Z_t$ as a vector-valued random process in $\mathbb{R}^d$ (in our particular example we have chosen $d = 8$, but let's be general here). So what has to change in the equations above in the vector case?  Dimensionally it's clear, that if $Z_t$ is a vector (and so therefore is $\textrm{d}Z_t$) then $f(Z_t, t)$ and $\textrm{d}W_t$ must be vectors also.  That's good, the vector-valued function $f$ still acts as the velocity, so it's natural for it to be a vector.  And the random component of the white noise contribution
$$
\textrm{d}W_t = \sqrt{\textrm{d}t}\, \xi_t,\qquad
$$
is drawn from the standard normal distribution on the vector space $\mathbb{R}^d$
$$
\xi_t \sim \mathcal{N}(0, I_d)\,.
$$
So each component in the vector just gets its own independent random step.

# Fokker-Planck equation

What about the Fokker-Planck equation?  The subject of that equation is the probability density function of the particles $u(z,t)$.  The function $u$ is still scalar-valued (it's a PDF, integrating to one as usual).  But its input $z$ is now a vector.  To account for this, the equation gets fancy vector differential operators in place of the scalar derivatives.  Here's the Fokker-Planck equation in $\mathbb{R}^d$:
$$
\frac{\partial u}{\partial t} = -\nabla \cdot (f u) + D\, \nabla^2 u\,.
$$
The first term on the right is the _divergence_, and is simply shorthand for
$$
\nabla \cdot (f u) = \sum_{i=1}^d \frac{\partial}{\partial z_i}\left(f_i\, u\right)\,.
$$
The second term on the right is the _Laplacian_, and is shorthand for
$$
\nabla^2 u = \sum_{i=1}^d \frac{\partial^2 u}{\partial {z_i}^2}\,.
$$
So written out in components, the right hand side of the Fokker-Planck equation is just the one-dimensional version summed over the $d$ components:
$$
\frac{\partial u}{\partial t} = \sum_{i=1}^d \left[-\frac{\partial}{\partial z_i}(f_i u) + D \frac{\partial^2 u} {\partial {z_i}^2}\right]\,.
$$

This reflects the fact that it's really just $d$ coordinates doing their own independent random walks subject to their own component of the velocity.


# Ornstein-Uhlenbeck process

In the specific case of the Ornstein-Uhlenbeck process

$$
\textrm{d}Z_{t} = -\theta Z_t\, \textrm{d}t + \sigma\, \textrm{d}W_t
$$

(which looks unchanged from last lesson, except remember $Z_t$ and $W_t$ are vectors now) the solution can still be found using Fourier transforms for a delta function initial condition $u(z,0) = \delta(z - z_0)$. It follows an isotropic Gaussian distribution

$$
Z_t \sim \mathcal{N}(\mu_t, s_t^2\,I_d)
$$

with mean

$$
\mu_t = z_0 \textrm{e}^{-\theta t}
$$

and covariance matrix $\Sigma = s_t^2 I_d$ where

$$
s_t^2 = \frac{D}{\theta}\left(1 - \textrm{e}^{-2\theta t}\right)\,.
$$

So, exactly like the one-dimensional case applied independently to each coordinate.  So far this transition to a vector-valued process is looking just fine!

If we like, we can write out the Gaussian PDF solution explicitly:
$$
u(z|z_0, t) = \frac{1}{(2\pi s_t^2)^{d/2}}\,\exp\left(-\frac{\|z - \mu_t\|^2}{2s_t^2}\right)\,.
$$

The solution for any initial condition can be found via convolution as usual
$$
u(z,t) = \int_{z_0} u(z|z_0,t)\, p_0(z_0)\, \mathrm{d}z_0
$$
where $u(z|z_0,t)$ is the Gaussian we wrote above.  This last equation will be important today.  Remember it's just a statement of the law of total probability: the unconditional distribution is found by integrating over all the conditional ones, weighted by the prior $p_0(z_0)$.

# Reverse process

The great swindle we pulled off in the last lesson to reverse the diffusion process continues to work.  We can take the Fokker-Planck equation for the Ornstein-Uhlenbeck process

$$
\frac{\partial u}{\partial t} = \nabla \cdot (\theta z u) + D\, \nabla^2 u\,,
$$

time-reverse it using $\tau = T - t$:

$$
\frac{\partial u}{\partial \tau} = -\nabla \cdot (\theta z u) - D\, \nabla^2 u\,,
$$

and with the same trickery as last time (exercises!), manipulate it into the form

$$
\frac{\partial u}{\partial \tau} = -\nabla \cdot \left(\left[\theta z +\frac{\sigma^2}{2} \nabla_z \log u \right]u\right)\,.
$$

The corresponding ODE for $z$ is then completely analogous to the result from last lesson

$$
\frac{\textrm{d}z}{\textrm{d}\tau} = \theta z + \frac{\sigma^2}{2} \nabla_z \log u
$$

once again involving the score function $\nabla_z \log u$.

So, on paper at least, everything looks under control in $d$ dimensions.  We can invert the Ornstein-Uhlenbeck process deterministically provided we know, or can at least approximate somehow, the score function $\nabla_z \log u$.

# Score function

But the more you think about it, the more it seems like a hopeless task to approximate the score function. We would need to somehow approximate $\nabla_z \log u$, where $u(z,t)$ is an unknown probability density on $d$-dimensional latent space that evolves with time.

So what to do?  Fortunately, we have a way to proceed.  We _don't_ try to learn $\nabla_z \log u$ directly.  Instead, we learn a way to undo _one step_ of the Ornstein-Uhlenbeck process at a time.  Here's how.

Start from the convolutional formula for $u$:
$$
u(z,t) = \int_{z_0} u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0\,.
$$

So the score function $\nabla_z \log u$ can be written
$$
\begin{align*}
\nabla_z \log u &= \frac{\nabla_z u(z,t)}{u(z,t)} \\
&= \frac{1}{u(z,t)} \int_{z_0} \nabla_z\, u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0\,.\qquad(*)
\end{align*}
$$


Now the gradient $\nabla_z\, u(z|z_0,t)$ is something we know about, because the form of $u(z|z_0,t)$ is Gaussian:
$$
u(z|z_0, t) = \frac{1}{(2\pi s_t^2)^{d/2}}\,\exp\left(-\frac{\|z - \mu_t\|^2}{2s_t^2}\right)\,.
$$

Taking the gradient, we see that
$$
\nabla_z u(z|z_0, t) = -\frac{z - \mu_t}{s_t^2}\, u(z|z_0, t)
$$

Substituting in $(*)$:
$$
\begin{align*}
\nabla_z \log u &= \frac{1}{u(z,t)} \int_{z_0} \nabla_z\, u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0 \\
&= \frac{-1}{u(z,t)} \int_{z_0} \frac{z - \mu_t}{s_t^2}\, u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0 \\
&= \frac{-1}{s_t^2\, u(z,t)}\left[ z \int_{z_0} u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0 - \int_{z_0} \mu_t\, u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0 \right] \\
&= \frac{-1}{s_t^2\, u(z,t)}\left[ z\, u(z,t) - \int_{z_0} \mu_t\, u(z|z_0,t)\, u_0(z_0)\, \mathrm{d}z_0 \right] \\
&= \frac{-1}{s_t^2}\left[ z - \int_{z_0} \frac{\mu_t\, u(z|z_0,t)\, u_0(z_0)}{u(z,t)}\, \mathrm{d}z_0 \right]\,. \\
\end{align*}
$$

Now for the Ornstein-Uhlenbeck process, $\mu_t = z_0 \textrm{e}^{-\theta t}$ and $s_t^2 = 1 - \textrm{e}^{-2\theta t}$.  So we get

$$
\begin{align*}
\nabla_z \log u &= \frac{-1}{1 - \textrm{e}^{-2\theta t}}\left[ z - \textrm{e}^{-\theta t} \int_{z_0} z_0\,\frac{u(z|z_0,t)\, u_0(z_0)}{u(z,t)}\, \mathrm{d}z_0 \right].
\end{align*}
$$

So far it's just been substituting in known formulas and rearranging.  But now look at the integral we've arrived at.  The fraction in the integrand is just Bayes' formula for the _conditional_ probability density
$$
u(z_0|z,t) := \frac{u(z|z_0,t)\, u_0(z_0)}{u(z,t)}\,.
$$
So the integral itself is the _expected value_
$$
\mathbb{E}[Z_0|Z_t,t] := \int_{z_0} z_0\,\frac{u(z|z_0,t)\, u_0(z_0)}{u(z,t)}\, \mathrm{d}z_0\,.
$$

Plugging this in, we arrive at
$$
\nabla_z \log u = \frac{-1}{1 - \textrm{e}^{-2\theta t}}\left( z - \textrm{e}^{-\theta t}\, \mathbb{E}[Z_0|Z_t,t] \right) \\
$$

At this point it's traditional to introduce the notation $$\alpha_t = \textrm{e}^{-2\theta t}\,,$$ and hence write the result as
$$
\nabla_z \log u = \frac{-1}{1 - \alpha_t}\left( z - \sqrt{\alpha_t}\, \mathbb{E}[Z_0|Z_t,t] \right)\,. \\
$$

Well now we're really getting somewhere.  Remarkably, the only quantity we actually need to estimate is this expected value $\mathbb{E}[Z_0|Z_t,t]$.  In words, we need to answer the question:

> Given a noise-corrupted latent $Z_t$ at simulation time $t$, what is the expected noise-free latent $Z_0$ it corresponds to?

That's it!  That's the question we need to train our neural net to answer.  And we can absolutely train it to answer that question, by feeding it lots and lots of true latents $Z_0$ (generated from real images) and their noise-corrupted versions $Z_t$ at all different times $t$.

# Generating training examples

It's even easier to generate suitable training examples $(Z_0, Z_t, t)$ than you might think. We don't even need to run the simulation forward in time, because we have the analytical solution for any time $t$:

$$
Z_t \sim \mathcal{N}(\mu_t,\, s_t^2\,I)
$$
which, in our new $\alpha_t$ notation is
$$
Z_t \sim \mathcal{N}(\sqrt{\alpha_t} Z_0,\, (1-\alpha_t) I)\,.
$$

So we can jump straight from $Z_0$ to a particular realisation $Z_t$ at any time $t$ by just sampling
$$
{\large \varepsilon} \sim \mathcal{N}(0, I)
$$
and computing
$$
Z_t = \sqrt{\alpha_t}\, Z_0 + \sqrt{1-\alpha_t}\, {\large \varepsilon}\,.\quad (\dagger)
$$

So to generate a training example, you start with an image of a digit.  You run it through the CVAE encoder to generate $Z_0$.  Then you choose a random value of $t \in (0,T]$, and you compute $Z_t$ using $(\dagger)$.  That's a training example.  Then you take another image of a digit. You run it through the CVAE encoder to generate $Z_0$, you choose a random $t \in (0, T]$ and compute $Z_t$ using $(\dagger)$.  That's another training example.  Repeat.
In this way you can generate as many training examples as you like.  It's fine and normal to even re-use the training _images_ multiple times, because although that will be a repeated value of $Z_0$, each time they'll be partnered with a different random value of $t$, and hence different $Z_t$.

# Training

Armed with your huge training set of  $(z_0^{(j)},\, z_t^{(j)},\, t_j)$ samples, you design some neural network model $M$, parameterised by $q$ (can't call the parameters $p$ since that symbol is taken), that takes as input $(z_t, t)$ and spits out its prediction of $z_0$:
$$
\hat{z_0} = M_q(z_t, t)\,.
$$

You train the model to minimise the **mean squared error (MSE)** between the model predictions $\hat{z_0}$ and the true noise-free latents $z_0$:
$$
L(q) = \frac{1}{N} \sum_{j=1}^{N} \left\|z_0^{(j)} - M_q(z_{t}^{(j)}, t_j)\right\|^2\,.
$$

Once trained, you now have a suitable approximation of the score function:
$$
\begin{align*}
\nabla_z \log u &= \frac{-1}{1 - \alpha_t}\left(z_t - \sqrt{\alpha_t}\, \mathbb{E}[Z_0|Z_t,t] \right) \\
&\approx \frac{-1}{1 - \alpha_t}\left(z_t - \sqrt{\alpha_t}\, M_q(z_t, t) \right)\,.
\end{align*}
$$


# Wait, what about labels?

Since we plan to use this score function in a generative model, surely we want a mechanism to specify what kind of image we want.  For example, if we want to generate an image of a digit 4, we should be able to specify "4" through a label somehow.

No problem!  The thing our model needs to learn is already the _conditional_ expectation $\mathbb{E}[Z_0|Z_t,t]$.  If we want to make it further conditioned on a label $y$, that's fine.  Just learn instead $\mathbb{E}[Z_0|Z_t,t,y]$ by training on a dataset of $(z_0^{(j)},\, z_t^{(j)},\, t_j,\, y_j)$ samples, with a model that predicts
$$
\hat{z}_0^{(j)} = M_q(z_t^{(j)},\, t_j,\, y_j)
$$
with loss function
$$
L(q) = \frac{1}{N} \sum_{j=1}^{N} \left\|z_0^{(j)} - M_q(z_{t}^{(j)},\, t_j,\, y_j)\right\|^2\,.
$$

The label $y$ just becomes an input to the model, same as time $t$.

# Wait, what's the architecture?
So sure, we need a neural net architecture that takes in $(z_t, t, y)$ and returns its prediction $\hat{z_0}$.  How, exactly?  Let's start with the inputs: how do we input the heterogeneous types $z_t \in \mathbb{R}^{d}$, $t \in \mathbb{R}$ and  $y \in \mathcal{L}$ to a neural network?  The first of these inputs is a $d$-dimensional vector, the second is a scalar, and the third is a label.

Our approach will be to run each of these inputs separately through a simple multi-layer perceptron (MLP) -- a couple of dense layers -- to expand them out to a common _feature size_.  We've opted for the swish activation layer once again.

We can use this MLP as-is for the latent $z_t$ and the time $t$.  The label $y$ will need to go through an embedding layer first, just as we did with tokens in our earlier language models.


In [ ]:
MLP(in_dim, feature_dim; hidden_dim = feature_dim) =
    Chain(
        Dense(in_dim => hidden_dim, swish),
        Dense(hidden_dim => feature_dim, swish)
    )

#
One we've transformed $z_t$, $t$ and $y$ to vectors of equal size, you might imagine we would stack them as three channels of a new array, and continue from there.  But experience shows that isn't necessary -- we can simply add them together to form a new vector!  It's the same idea we used for the positional embedding in our GPT-2 model from lesson 6.

We call our model `DDIMNet` for denoising diffusion implicit model, although the [original paper](https://arxiv.org/pdf/2010.02502) applied the process directly to image space, rather than to a latent space as is done in more modern approaches.

In [ ]:
DDIMNet(latent_dim; hidden_dim=256, num_classes=10) = @compact(
    mlp_z = MLP(latent_dim, hidden_dim),        # z ∈ Rᵈ
    mlp_t = MLP(1, hidden_dim),                 # t ∈ R
    mlp_y = Chain(                              # y ∈ {1,..,num_classes}
      Embedding(num_classes => hidden_dim),
      MLP(hidden_dim, hidden_dim)),
    block = MLP(hidden_dim, hidden_dim),        # processing MLP
    out = Dense(hidden_dim => latent_dim),      # output
) do (z,t,y)

    hz = mlp_z(z)
    ht = mlp_t(t)
    hy = mlp_y(y)

    h = hz + ht + hy
    b = block(h)
    @return out(b)
end

ddim_net = DDIMNet(8)

#
This is trainable on a CPU, so let's do that now.  We'll first encode all of our MNIST images to latent vectors to use as our training set.  Then load them all into a `DataLoader` to handle automatic batching and shuffling.

In [ ]:
z = zeros(Float32, 8, size(images, 4))

# Encoding in batches is faster
for k = 1:1000:size(images,4)
  z[:,k:k+999] = encode(images[:,:,:,k:k+999])
end

data_loader = DataLoader((z, labels); batchsize = 64, shuffle = true, partial = false)

#
Initialise the network parameters.

In [ ]:
rng = Random.default_rng()
p_ddim0, st_ddim0 = Lux.setup(rng, ddim_net);

#
And here is the full code to train the model.  You can see how each new training example is computed according to
$$
Z_t = \sqrt{\alpha_t}\, Z_0 + \sqrt{1-\alpha_t}\, {\large \varepsilon}\,.\quad (\dagger)
$$

We've chosen $\theta = 1/2$, so that

$$\alpha_t = \textrm{e}^{-2\theta t} = \textrm{e}^{-t}\,.$$

In [ ]:
# The full code to train the model.
function train(; epochs, T)

    # Mean squared error loss, Adam optimiser
    mseloss = MSELoss()
    autodiff = AutoZygote()
    trainer = Lux.Training.TrainState(ddim_net, p_ddim0, st_ddim0, Lux.Optimisers.Adam())

    # Main training loop
    for epoch = 1:epochs

        total_loss = 0.0f0
        nbatches = 0

        # Process the data by batch
        for (z0, labels) in data_loader

            y = labels .+ 1                           # labels are indices 1-10
            t = T * rand_like(y', Float32)            # Uniform(0,T) random time for each sample
            ε = randn_like(z0)                        # Normal(0,1) random noise for each sample
            α = exp.(-t)                              # since θ = 1/2
            zt = sqrt.(α) .* z0 + sqrt.(1 .- α) .* ε  # noisify each sample according to α

            # Training step, with MSE loss computed between predicted z0 and true z0
            g, lossval, extras, trainer = Training.compute_gradients(
                  autodiff, mseloss, ((zt, t, y), z0), trainer
            )
            trainer = Training.apply_gradients(trainer, g)

            total_loss += lossval
            nbatches += 1

        end

        progress = (; epoch, mean_loss = total_loss / nbatches)
        @show progress

    end

    return trainer

end

# The final time T

We still need to address the question of how long is "long enough" -- that is, how do we choose the value for the final time $T$.

Recall the explicit formula $(\dagger)$ for the forward noisification process:
$$
Z_t = \sqrt{\alpha_t}\, Z_0 + \sqrt{1-\alpha_t}\, {\large \varepsilon}
$$
where $\alpha_t = \textrm{e}^{-t}$ for $\theta = 1/2$.  So the original noise-free latent $Z_0$ is being "faded out" as $\sqrt{\alpha_t} \to 0$ and the noise $\large \varepsilon$ is being "faded in" as $\sqrt{1 - \alpha_t} \to 1$.  We can compute the signal to noise ratio (SNR) as
$$
\textrm{SNR}(t) = \frac{\textrm{Var}[\textrm{signal}]}{\textrm{Var}[\textrm{noise}]} = \frac{\textrm{Var}[\sqrt{\alpha_t}Z_0]}{\textrm{Var}[\sqrt{1 - \alpha_t}{\Large \varepsilon}]} = \frac{\alpha_t}{1 - \alpha_t} = \frac{\textrm{e}^{-t}}{1-\textrm{e}^{-t}}
$$
where by "signal" we mean the original noise-free latent, and we are assuming $\textrm{Var}[Z_0] = \textrm{Var}[{\Large \varepsilon}] = 1$.

So as $t \to \infty$, $\textrm{SNR(t)} \approx \textrm{e}^{-t}$. If we judge the latent vector to have been turned into pure noise when the signal-to-noise ratio is less than 1%, we would require
$$
\textrm{SNR(t)} \approx \textrm{e}^{-t} < 0.01 \implies t > 4.6.
$$

So $T = 5$ is a perfectly reasonable choice.

#
Let's train the model for 10 epochs, which should be enough to see it working. It's not a big model to train, so 10 epochs should only take a couple of minutes on the CPU.  This is a big advantage of working in latent space: most of the heavy lifting was done in training the CVAE encoder/decoder pair.  Once they're trained, the diffusion component is not a big deal for us.

In [ ]:
@time trainer = train(epochs = 10, T = 5f0);
p_ddim = trainer.parameters;
st_ddim = trainer.states;

#
We'll make our usual inference helper function.

In [ ]:
model(z, t, y) = first(ddim_net((z, t, y.+1), p_ddim, st_ddim))

# Digit archetypes

As a quick sanity check, the model should return a plausible noise-free latent $\hat{z_0}$ when given sensible inputs $(z_t, t, y)$. Remember the neural net's job is literally to predict $\mathbb{E}[Z_0|Z_t,t,y]$: the expected noise-free latent $Z_0$, for any noisified latent $Z_t$ at time $t$ and label $y$. So let's try it out with _pure random noise_ $z_t$, corresponding to $t = T$, and we'll condition on each label $y \in \{0,\ldots,9\}$.

In [ ]:
z0 = model(randn(Float32, 8, 10), fill(5f0, 1, 10), 0:9)

#
Viewing these decoded images reveals the digit archetypes that the model learned during training.  These are the model's "one shot" predictions conditioned on a label, with pure random noise as input.

They look great!  Having seen many different examples of each digit during training, it's picked out very clean archetypal versions of each digit.  These are basically independent of the noise provided (try it!) -- it's mostly the label conditioning that's driving these predictions.

In [ ]:
imgs = decode(z0)

function display_images(imgs)
    fig = Figure(size=(800,400))
    for i = 1:5
        image(fig[1,i], imgs[:,:,1,i], axis=(aspect=1, title="$(i-1)")) |> first |> hidedecorations!
        image(fig[2,i], imgs[:,:,1,i+5], axis=(aspect=1, title="$(i+4)")) |> first |> hidedecorations!
    end
    fig
end

display_images(imgs)

#
However, it's essential that the value of $t$ we provide is consistent with the input $Z_t$.  Remember the model was trained on the forward Ornstein-Uhlenbeck process, starting from a noise-free latent $Z_0$.  As time $t \to \infty$ that latent devolves into pure noise.  So only a sufficiently large $t$ is consistent with an input of pure noise, $Z_t \sim \mathcal{N}(0,I)$.  If we pass it pure noise latents but an early time, say $t = 0.1$, it will be confused.

In [ ]:
# oops t = 0.1 instead of 5
z0 = model(randn(Float32, 8, 10), fill(0.1f0, 1, 10), 0:9)
imgs = decode(z0);
display_images(imgs)

# Reverse diffusion for real

Anyway, "one-shotting" the prediction like this is not how we're supposed to actually run the model.  It's just a quick check that it seems to be performing correctly.

In case we've forgotten, the real point of building this model was to approximate the score function $\nabla_z\log u$ for the reverse diffusion process

$$
\frac{\textrm{d}z}{\textrm{d}\tau} = \theta z + \frac{\sigma^2}{2} \nabla_z\log u(z,T-\tau)\,,\quad  0 < \tau \leq T
$$
subject to
$$
z_{\tau = 0} \sim \mathcal{N}(0,I)\,.
$$

We have worked hard to derive the elegant formula for the score function
$$
\nabla_z \log u = \frac{-1}{1 - \alpha_t}\left(z_t - \sqrt{\alpha_t}\, \mathbb{E}[Z_0|Z_t,t,y] \right)
$$
and our model $M_q$ approximates the conditional expectation $\mathbb{E}[Z_0|Z_t,t,y]$.  So now we just need to define the score function
$$
\textrm{score}(z,t,y) = \frac{-1}{1 - \alpha_t}\left(z - \sqrt{\alpha_t}\, M_q(z, t, y) \right)\,.
$$

In [ ]:
function score(z, t, y)
    nsamples = length(y)
    α = exp.(-t)
    ts = fill(t, 1, nsamples)
    z0 = model(z, ts, y)
    return -1f0./(1f0.-α) * (z - sqrt(α)*z0)
end

# SDE simulator

Here's our stochastic differential equation simulator from last lesson.  We made it dimension agnostic already, so it will handle our latent variables now being vectors in 8-dimensional space.

In [ ]:
function simulate_sde(; z0, nsteps, T, f = (z,t) -> 0*z, σ = 1)

    Z = stack(fill(z0, nsteps+1))

    dt = T / nsteps  # timestep

    for n = 1:nsteps
        tₙ = (n-1) * dt
        zₙ = selectdim(Z, ndims(Z), n)
        zₙ₊₁ = selectdim(Z, ndims(Z), n+1)
        ξ = randn_like(zₙ)
        dW = sqrt(dt) * ξ
        zₙ₊₁ .= zₙ + f(zₙ,tₙ) * dt + σ * dW
    end

    return Z
end

# Simulation time

Well what are we waiting for?  Let's get simulating!  We're starting from pure random noise in latent space, and running diffusion in reverse until the "particles" have arranged themselves according to our model of the latent image distribution.

The output is the full trajectory over the simulation for each "particle".

In [ ]:
nsteps = 100
T = 5f0
y = 0:4
ndigits = length(y)
z0 = randn(Float32, latent_dim, ndigits)
@time Z = simulate_sde(; z0, nsteps, T, f = (z,τ) -> z/2 + score(z,T-τ,y)/2, σ = 0)
size(Z)

#
So we can see how each digit was materialised from its random starting point by decoding the history of the latent trajectories.

Note that it's always possible that by chance the starting position in latent space corresponds closely to the latent representation of an actual digit.  Maybe even one with the correct label.  In other cases the starting position will not correspond to the latent representation of a digit, and hence won't decode to anything digit-like at first.  But no matter, after the reverse diffusion process has done its work, all the "particles" will have wandered their way into positions that do correspond to actual latent images, consistent with their labels.

The final column ($t = 0$) corresponds to the fully denoised latents, which all look consistent with their requested labels 0,1,2,3,4.

In [ ]:
fig = Figure(size=(800,500))
idxs = [1 61 81 91 96 98 100 101]
for i = 1:ndigits, (pos,j) in enumerate(idxs)
    t = round(5-(j-1)/20, digits=2)
    ax, _ = image(fig[i,pos], decode(Z[:,i,j])[:,:,1,1], axis=(aspect=1, title="t=$t"))
    hidedecorations!(ax)
end
fig

#
Remember we are running a deterministic simulation here.  So for identical noise inputs, and identical labels, the image will come out the same every time.  But if you choose different random noise inputs, you will get different realisations of the same digit.  Here's 5 different trajectories for generating the digit 5, for example.

In [ ]:
nsteps = 100
T = 5f0
digit = 5
y = fill(digit, ndigits)
ndigits = length(y)
z0 = randn(Float32, latent_dim, ndigits)
@time Z = simulate_sde(; z0, nsteps, T, f = (z,τ) -> z/2 + score(z,T-τ,y)/2, σ = 0)

fig = Figure(size=(800,500))
idxs = [1 61 81 91 96 98 100 101]
for i = 1:ndigits, (pos,j) in enumerate(idxs)
    t = round(5-(j-1)/20, digits=2)
    ax, _ = image(fig[i,pos], decode(Z[:,i,j])[:,:,1,1], axis=(aspect=1, title="t=$t"))
    hidedecorations!(ax)
end
fig

# Stable diffusion
Industry grade methods like Stable Diffusion operate on the same principle, albeit with many improvements. For working with detailed colour images, the latent representation is generally chosen to be something like $z \in \mathbb{R}^{64 \times 64 \times 4}$ rather than just $z \in \mathbb{R}^d$.  So the VAE encodes images into a compressed, but still spatial, representation, which becomes the input $z$ to the diffusion model.

The text prompt is tokenised and passed through a transformer-based text encoder, similar to what we learned about for GPT.  Its job isn't next token prediction though; its job is to turn text prompts into helpful embeddings that will guide the image generation. It could be a model that's trained in advance for this purpose, like [CLIP](https://en.wikipedia.org/wiki/Contrastive_Language-Image_Pre-training).  Or it could be a bespoke model which is trained jointly with the diffusion model, as we did. It provides the label input $y$ to the diffusion model.

The time variable input goes through its own encoding, either learned outright as we did, or incorporating [positional encoding](https://en.wikipedia.org/wiki/Transformer_(deep_learning)#Positional_encoding), to become the input $t$ to the score function.

The diffusion model for the score function incorporates a so-called [U-net](https://en.wikipedia.org/wiki/U-Net) architecture which includes:
* convolutions (to exploit local spatial structure of the latent image)
* self-attention (to allow distant regions of the latent image to exchange information)
* cross-attention between the embedded prompt tokens and the latent image features (to allow the prompt to influence the generated image)

On the last point, cross-attention, it's worth remembering that the attention mechanism uses query vectors $q^{(j)}$, key vectors $k^{(i)}$ and value vectors $v^{(i)}$.  That's it.  It's perfectly fine for the query vectors to come from the latent image (via a weight matrix $W_Q$), and for the key vectors and value vectors to come from the prompt tokens (via weight matrices $W_K$ and $W_V$).  You just arrange all the dimensions for the weight matrices so that the resulting vectors $q^{(j)}$, $k^{(i)}$ and $v^{(i)}$ all belong to the same space, then attention works just fine.


# Conclusion

In this lesson we learned:

* the Ornstein-Uhlenbeck process in $\mathbb{R}^d$

* how to write its score function in terms of the conditional expectation $\mathbb{E}[Z_0 | Z_t, t]$

* how this leads to a practical training approach in which a neural network learns to predict clean latents from noisy ones

* how label information can be included so that the reverse process generates digits of a chosen class

* how repeated denoising steps in latent space can produce new MNIST-style samples from pure noise